In [1]:
from dotenv import load_dotenv

load_dotenv('E:/Courses/Intro to Langchain/lca-lc-foundations-main/example.env')

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the count directly. Could you tell me which database or data source you’re using and what you want counted (total rows vs. distinct artists)? Here are common options you can run:

- SQL (assuming a table named artists)
  - Total rows: SELECT COUNT(*) AS artist_count FROM artists;
  - Distinct artists by id: SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artists;
  - Distinct artists by name: SELECT COUNT(DISTINCT name) AS artist_count FROM artists;

- MongoDB (collection named artists)
  - Total documents: db.artists.countDocuments({})
  - Distinct artists by a field (e.g., artist_id): db.artists.distinct("artist_id").length

- CSV/Excel (loaded with Python/pandas)
  - import pandas as pd
    df = pd.read_csv("artists.csv")
    total = len(df)
    distinct = len(df['artist_id'].drop_duplicates())

If you share your setup (database type, table/collection name, and relevant schema), I’ll give you the exact command or query to ru